# Build CVAE Tensor Cache

CPU preprocess가 만든 manifest/stats를 이용해 split별 tensor cache를 만든다. 이후 GPU 학습은 작은 npz 수천 개를 열지 않고 이 cache만 읽는다.


In [ ]:
# CPU runtime에서 실행
%pip install -q numpy pandas torch tqdm

from __future__ import annotations
from pathlib import Path
import gc
import json
import os
import shutil
import subprocess

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm


In [ ]:
# Drive setup + local SSD staging
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

def find_filled_samples_root(drive_root):
    candidates = [
        drive_root / 'filled_samples',
        drive_root / 'attempt4' / 'cloud_fill_residual_v1' / 'filled_samples',
        drive_root / 'cloud_fill_residual_v1' / 'filled_samples',
    ]
    for cand in candidates:
        count = len(list(cand.glob('*/*.npz'))) if cand.exists() else 0
        print('filled_samples candidate:', cand, 'count=', count)
        if count > 0:
            return cand.resolve(), count
    raise FileNotFoundError('filled_samples not found on Drive')

def copy_dir_to_local(src, expected_count, dst):
    dst = Path(dst)
    existing = len(list(dst.glob('*/*.npz'))) if dst.exists() else 0
    if existing == expected_count:
        print('local samples already staged:', dst, 'count=', existing)
        return dst.resolve()
    if dst.exists():
        shutil.rmtree(dst)
    dst.mkdir(parents=True, exist_ok=True)
    print('copy Drive filled_samples -> local SSD')
    print('src:', src)
    print('dst:', dst)
    result = subprocess.run(['bash', '-lc', f'rsync -a --info=progress2 "{src}/" "{dst}/"'], text=True)
    if result.returncode != 0:
        print('rsync failed; fallback to shutil.copytree')
        shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(src, dst)
    copied = len(list(dst.glob('*/*.npz')))
    print('local copied count:', copied)
    if copied != expected_count:
        raise RuntimeError(f'local copy count mismatch: copied={copied}, expected={expected_count}')
    return dst.resolve()

DRIVE_ROOT = Path('/content/drive/MyDrive') if IN_COLAB else Path(os.environ.get('DRIVE_ROOT', '/content/drive/MyDrive'))
DRIVE_SAMPLE_DIR, sample_count = find_filled_samples_root(DRIVE_ROOT)
FILL_RUN_DIR = DRIVE_SAMPLE_DIR.parent
PREPROCESS_DIR = FILL_RUN_DIR / 'cvae_preprocess_v4'
CACHE_DIR = FILL_RUN_DIR / 'cvae_tensor_cache_v4'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_DIR = copy_dir_to_local(DRIVE_SAMPLE_DIR, sample_count, Path('/content/filled_samples'))
print('SAMPLE_DIR local:', SAMPLE_DIR)
print('PREPROCESS_DIR:', PREPROCESS_DIR)
print('CACHE_DIR:', CACHE_DIR)


In [ ]:
manifest_path = PREPROCESS_DIR / 'dataset_manifest.csv'
stats_path = PREPROCESS_DIR / 'normalization_stats_cvae_v4.json'
validation_path = PREPROCESS_DIR / 'dataset_validation.csv'
for p in [manifest_path, stats_path, validation_path]:
    if not p.exists():
        raise FileNotFoundError(f'missing CPU preprocess output: {p}. Run 02_0 first.')

manifest_df = pd.read_csv(manifest_path)
stats = json.loads(stats_path.read_text(encoding='utf-8'))
CONDITION_CHANNELS = [str(x) for x in stats['condition_channels']]
TARGET_CHANNELS = [str(x) for x in stats['target_channels']]
ALL_TARGET_CHANNELS = [str(x) for x in stats['source_target_channels']]
TARGET_INDICES = [int(x) for x in stats['target_indices']]
META_CHANNELS = [str(x) for x in stats['meta_channels']]
print('manifest counts:', manifest_df.groupby('split').size().to_dict())
print('condition channels:', len(CONDITION_CHANNELS))
print('target channels:', TARGET_CHANNELS)
print('meta channels:', META_CHANNELS)


In [ ]:
CACHE_DTYPE = np.float16  # normalized tensors라 fp16 cache로 IO/RAM 절약

def as_2d_mask(arr):
    arr = np.asarray(arr)
    if arr.ndim == 3:
        return arr[0].astype(bool)
    return arr.astype(bool)

def minmax(values, channel_stats, names):
    out = values.astype(np.float32, copy=True)
    for i, name in enumerate(names):
        lo = channel_stats[name]['min']; hi = channel_stats[name]['max']
        denom = hi - lo
        out[i] = 0.0 if abs(denom) <= 1e-12 else (out[i] - lo) / denom
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def select_target_and_mask(z):
    target_all = z['target'].astype(np.float32)
    mask_all = z['target_mask'].astype(bool)
    return target_all[TARGET_INDICES], mask_all[TARGET_INDICES]

def select_target_weight(z, target_mask):
    if 'target_weight' in z.files:
        weight = z['target_weight'].astype(np.float32)[TARGET_INDICES]
        return np.where(target_mask, weight, 0.0).astype(np.float32)
    if 'cloud_fill_uncertainty' in z.files:
        unc = z['cloud_fill_uncertainty'].astype(np.float32)
        if unc.ndim == 2:
            unc = unc[None]
        weight = 1.0 / (1.0 + unc[:1] / 5.0)
        clear = as_2d_mask(z['clear_mask'])
        weight[:, clear] = 1.0
        return np.where(target_mask, weight, 0.0).astype(np.float32)
    return target_mask.astype(np.float32)

def local_path(rel):
    return SAMPLE_DIR / str(rel)


In [ ]:
def build_split_cache(split):
    rows = manifest_df[manifest_df['split'] == split].reset_index(drop=True)
    if rows.empty:
        print('skip empty split:', split)
        return None
    first_path = local_path(rows.loc[0, 'file'])
    with np.load(first_path, allow_pickle=True) as z:
        cond0 = minmax(z['condition'].astype(np.float32), stats['condition'], CONDITION_CHANNELS)
        target0, mask0 = select_target_and_mask(z)
        target0 = minmax(target0, stats['target'], TARGET_CHANNELS)
        weight0 = select_target_weight(z, mask0)
        clear0 = as_2d_mask(z['clear_mask']).astype(bool)[None]
        cloud0 = as_2d_mask(z['cloud_fill_mask']).astype(bool)[None]
        meta0 = z['meta'].astype(np.float32)
    n = len(rows)
    condition = np.empty((n, *cond0.shape), dtype=CACHE_DTYPE)
    target = np.empty((n, *target0.shape), dtype=CACHE_DTYPE)
    target_mask = np.empty((n, *mask0.shape), dtype=np.bool_)
    target_weight = np.empty((n, *weight0.shape), dtype=CACHE_DTYPE)
    clear_mask = np.empty((n, *clear0.shape), dtype=np.bool_)
    cloud_fill_mask = np.empty((n, *cloud0.shape), dtype=np.bool_)
    meta = np.empty((n, *meta0.shape), dtype=np.float32)
    files = []
    nonfinite_repairs = []
    for i, row in tqdm(rows.iterrows(), total=n, desc=f'build cache {split}'):
        rel = row['file']
        p = local_path(rel)
        with np.load(p, allow_pickle=True) as z:
            raw_condition = z['condition'].astype(np.float32)
            bad_pixels = int((~np.isfinite(raw_condition)).sum())
            if bad_pixels:
                bad_by_ch = ~np.isfinite(raw_condition).reshape(raw_condition.shape[0], -1)
                bad_names = [CONDITION_CHANNELS[j] for j in np.where(bad_by_ch.any(axis=1))[0]]
                nonfinite_repairs.append({'file': rel, 'condition_nonfinite_pixels': bad_pixels, 'condition_nonfinite_channels': '|'.join(bad_names)})
            c = minmax(raw_condition, stats['condition'], CONDITION_CHANNELS)
            t, m = select_target_and_mask(z)
            w = select_target_weight(z, m)
            t = minmax(t, stats['target'], TARGET_CHANNELS)
            condition[i] = c.astype(CACHE_DTYPE)
            target[i] = t.astype(CACHE_DTYPE)
            target_mask[i] = m.astype(np.bool_)
            target_weight[i] = np.nan_to_num(w, nan=0.0, posinf=0.0, neginf=0.0).astype(CACHE_DTYPE)
            clear_mask[i] = as_2d_mask(z['clear_mask']).astype(bool)[None]
            cloud_fill_mask[i] = as_2d_mask(z['cloud_fill_mask']).astype(bool)[None]
            meta[i] = np.nan_to_num(z['meta'].astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
        files.append(str(rel))
    cache = {
        'split': split,
        'files': files,
        'condition': torch.from_numpy(condition),
        'target': torch.from_numpy(target),
        'target_mask': torch.from_numpy(target_mask),
        'target_weight': torch.from_numpy(target_weight),
        'clear_mask': torch.from_numpy(clear_mask),
        'cloud_fill_mask': torch.from_numpy(cloud_fill_mask),
        'meta': torch.from_numpy(meta),
        'stats': stats,
        'condition_channels': CONDITION_CHANNELS,
        'target_channels': TARGET_CHANNELS,
        'source_target_channels': ALL_TARGET_CHANNELS,
        'target_indices': TARGET_INDICES,
        'meta_channels': META_CHANNELS,
        'cache_dtype': str(CACHE_DTYPE),
    }
    out_path = CACHE_DIR / f'cvae_cache_{split}.pt'
    torch.save(cache, out_path)
    print('saved:', out_path, 'size_gb=', out_path.stat().st_size / (1024**3))
    if nonfinite_repairs:
        rep_path = CACHE_DIR / f'condition_nonfinite_repairs_{split}.csv'
        pd.DataFrame(nonfinite_repairs).to_csv(rep_path, index=False, encoding='utf-8-sig')
        print('nonfinite condition repaired in cache:', len(nonfinite_repairs), rep_path)
    del cache, condition, target, target_mask, target_weight, clear_mask, cloud_fill_mask, meta
    gc.collect()
    return out_path

cache_paths = {}
for split in ['train', 'val', 'test']:
    out = build_split_cache(split)
    if out is not None:
        cache_paths[split] = str(out)
summary = {
    'cache_dir': str(CACHE_DIR),
    'sample_dir': str(DRIVE_SAMPLE_DIR),
    'preprocess_dir': str(PREPROCESS_DIR),
    'cache_paths': cache_paths,
    'counts': manifest_df.groupby('split').size().astype(int).to_dict(),
    'condition_channels': CONDITION_CHANNELS,
    'target_channels': TARGET_CHANNELS,
    'meta_channels': META_CHANNELS,
}
summary_path = CACHE_DIR / 'cvae_tensor_cache_summary.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print('cache build done:', summary_path)
